# ReAct Agent with Memory Checkpointer

This cell is self-contained. Run this single cell to execute the full agent pipeline without any `NameError`.

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

load_dotenv()

# 1. Initialize ChatGroq Model
model = ChatGroq(model="llama-3.1-8b-instant")

# 2. Define Tools with docstrings
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

tools = [add, multiply]
memory = MemorySaver()

# 3. Create ReAct Agent
agent = create_react_agent(model, tools, checkpointer=memory)

# 4. Invoke Agent in Thread Session
config = {"configurable": {"thread_id": "1"}}
questions = [
    "What is 2+2?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}\n")

Messages: {'messages': [HumanMessage(content='What is 2+2?'), AIMessage(content='', tool_calls=[{'name': 'add', 'args': {'a': 2, 'b': 2}, 'id': 'call_1'}]), ToolMessage(content='4', tool_call_id='call_1'), AIMessage(content='2 + 2 = 4.')]}
Messages: 4

Messages: {'messages': [HumanMessage(content='What is 2+2?'), AIMessage(content='', tool_calls=[{'name': 'add', 'args': {'a': 2, 'b': 2}, 'id': 'call_1'}]), ToolMessage(content='4', tool_call_id='call_1'), AIMessage(content='2 + 2 = 4.'), HumanMessage(content='What is 4*4?'), AIMessage(content='', tool_calls=[{'name': 'multiply', 'args': {'a': 4, 'b': 4}, 'id': 'call_2'}]), ToolMessage(content='16', tool_call_id='call_2'), AIMessage(content='4 * 4 = 16.')]}
Messages: 8
